In [7]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/syahribanun/fp-nlp/dataset_dedup.jsonl
/kaggle/input/datasets/syahribanun/fp-nlp/holdout.jsonl


# Install

In [8]:
!pip install -q -U transformers accelerate
!pip install -q antlr4-python3-runtime==4.11   # utk sympy parse_latex

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/kaggle/working/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO); os.chdir(REPO)
print('repo ready:', REPO)

In [ ]:
import glob
# Skenario 4 — model pembanding (baseline) di TEST SET YANG SAMA dgn S2/S3 (data/sft/test/).
# Test set sama -> baseline (S4) langsung comparable dgn model kita (S2/S3). Metrik sama: sampling
# N=5/soal -> pass@1,2,3 + maj@3,5.
def find_set(keyword):
    for pat in [f'/kaggle/input/**/*{keyword}*test*.jsonl', f'data/sft/test/*{keyword}*.jsonl']:
        hits = glob.glob(pat, recursive=True)
        if hits: return hits[0]
    return f'data/sft/test/{keyword}_test.jsonl'

SET_PATHS = {'numglue': find_set('numglue'), 'un': find_set('un'), 'easy': find_set('easy')}
OUT = 'data/eval/s4_baselines.json'
MAX_NEW_TOKENS = 4096   # reasoning model (OpenMath) butuh panjang
METRICS = ['pass@1', 'pass@2', 'pass@3', 'maj@3', 'maj@5']

MODELS = {
    'OpenMath-Nemotron-1.5B': 'nvidia/OpenMath-Nemotron-1.5B',
    'SeaLLMs-v3-1.5B-Chat'  : 'SeaLLMs/SeaLLMs-v3-1.5B-Chat',
    'Qwen2.5-1.5B-Instruct' : 'Qwen/Qwen2.5-1.5B-Instruct',
}
print('test sets:', SET_PATHS)

# Load Test Set (sama dgn S2/S3)

In [ ]:
from src.eval.scenario_eval import load_sets

# Test set sama dgn S2/S3 (numglue/un/easy). Yang filenya tak ada -> dilewati otomatis.
sets = load_sets(SET_PATHS)
# Baseline = HF model id, tanpa adapter LoRA.
specs = {name: {'model': mid, 'adapter': None} for name, mid in MODELS.items()}
print('sets:', {k: len(v) for k, v in sets.items()})
print('models:', list(specs))

# Eval Model

In [ ]:
from src.eval.sample_eval import eval_specs_sampling

# Sampling N=5/soal (temp 0.7) -> pass@1,2,3 + maj@3,5 (sama persis dgn S2).
results = eval_specs_sampling(specs, sets, n_samples=5, ks_pass=(1, 2, 3), ks_maj=(3, 5),
                             temperature=0.7, top_p=0.95,
                             max_new_tokens=MAX_NEW_TOKENS, batch_size=4)
print('done')

# Comparison

In [ ]:
from src.eval.sampling_metrics import render_tables
import json

print('\n=== SKENARIO 4 — baseline di test set (sama dgn S2/S3) ===')
print(render_tables(results, METRICS))

Path('data/eval').mkdir(parents=True, exist_ok=True)
Path(OUT).write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nsummary ->', OUT)